# UTS Graf Sains Data — Bagian 2.2
## Analytics

### Tujuan
Analisis ini bertujuan untuk membangun sistem rekomendasi item menggunakan metode K-Nearest Neighbors (KNN) berdasarkan pola interaksi pengguna pada dataset RetailRocket. Selanjutnya, hasil rekomendasi dievaluasi menggunakan metrik Hit@5 dan Normalized Discounted Cumulative Gain (NDCG@5).

### Periode analisis
- **P1 (Training):** 1–15 Juli 2015.
- **P2 (Testing):** 16–31 Agustus 2015.

### Metode analisis
1. Memilih satu user secara acak dengan proses yang dapat direproduksi.
2. Menerapkan KNN dengan \(k=5\) untuk mencari pengguna yang memiliki pola interaksi serupa.
3. Menghasilkan rekomendasi item dan menampilkan tiga item yang direkomendasikan.
4. Menghitung Hit@5 dan NDCG@5 berdasarkan interaksi pada periode pengujian.

### Sumber data
Dataset RetailRocket yang telah dipisahkan menjadi `P1.csv` dan `P2.csv` pada Bagian 2.1.

# Import library dan membaca data P1 serta P2

In [1]:
# UTS Graf Sains Data — Bagian 2.2 — STEP 1: Import library dan membaca dataset
import os
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors

# Parameter
SEED = 42            # seed pemilihan user acak -> hasil reproducible
K = 5                # jumlah tetangga KNN
TOP_N = 5            # panjang daftar rekomendasi (untuk Hit@5 dan NDCG@5)


P1_FILE = os.path.join("data", "P1.csv")
P2_FILE = os.path.join("data", "P2.csv")

df_p1 = pd.read_csv(P1_FILE)
df_p2 = pd.read_csv(P2_FILE)
print("DATA TRAINING (P1):", df_p1.shape)
print("DATA TESTING  (P2):", df_p2.shape)


DATA TRAINING (P1): (333778, 5)
DATA TESTING  (P2): (287853, 5)


Pada tahap ini, dilakukan import library yang dibutuhkan untuk membangun sistem rekomendasi menggunakan metode K-Nearest Neighbors (KNN). Library pandas digunakan untuk membaca dan mengolah dataset, scipy.sparse untuk menyimpan matriks interaksi secara hemat memori, sedangkan scikit-learn digunakan untuk normalisasi data dan mencari tetangga terdekat.

Dataset dibagi menjadi dua periode, yaitu P1 sebagai data training dan P2 sebagai data testing. Hasil pembacaan menunjukkan bahwa data training memiliki 333.778 baris dan 5 kolom, sedangkan data testing memiliki 287.853 baris dan 5 kolom.

Parameter SEED = 42 digunakan agar pemilihan pengguna secara acak dapat diulang dengan hasil yang konsisten. Parameter K = 5 menentukan jumlah tetangga yang digunakan dalam KNN, sedangkan TOP_N = 5 menentukan jumlah item teratas yang dievaluasi untuk metrik Hit@5 dan NDCG@5.

Dengan demikian, dataset dan parameter awal telah disiapkan untuk proses pembangunan dan evaluasi sistem rekomendasi pada tahap berikutnya.

# Menyiapkan bobot interaksi pengguna

Bobot ini merupakan pilihan desain untuk sistem rekomendasi: view diberi bobot 1, addtocart bobot 3, dan transaction bobot 5. Artinya, aktivitas menambahkan item ke keranjang dan melakukan transaksi dianggap menunjukkan ketertarikan yang lebih kuat dibandingkan sekadar melihat item.

In [2]:
# 2.2.2 — Memberikan bobot pada jenis interaksi

BOBOT_EVENT = {
    "view": 1.0,
    "addtocart": 3.0,
    "transaction": 5.0
}

# Pastikan item dan visitor memiliki ID numerik
for df in [df_p1, df_p2]:
    df["visitorid"] = pd.to_numeric(
        df["visitorid"], errors="coerce"
    )
    df["itemid"] = pd.to_numeric(
        df["itemid"], errors="coerce"
    )
    df.dropna(
        subset=["visitorid", "itemid", "event"],
        inplace=True
    )
    df["visitorid"] = df["visitorid"].astype(int)
    df["itemid"] = df["itemid"].astype(int)
    df["event"] = df["event"].astype(str).str.lower()

# Terapkan bobot pada data training P1
df_p1["bobot"] = df_p1["event"].map(BOBOT_EVENT).fillna(0.0)

print("Distribusi event pada P1:")
display(df_p1["event"].value_counts().rename_axis("event").reset_index(name="jumlah"))

print("\nContoh data training dengan bobot:")
display(df_p1[["visitorid", "itemid", "event", "bobot"]].head(10))


Distribusi event pada P1:


,event,jumlah
0,view,322990
1,addtocart,8099
2,transaction,2689



Contoh data training dengan bobot:


,visitorid,itemid,event,bobot
0,457980,278224,view,1.0
1,554795,6605,view,1.0
2,1291838,444688,view,1.0
3,567028,296749,view,1.0
4,174978,99274,view,1.0
5,686848,352723,view,1.0
6,346426,212672,view,1.0
7,957413,58638,view,1.0
8,459541,171711,view,1.0
9,1048488,170536,view,1.0


Pada tahap ini, setiap jenis interaksi pengguna terhadap item diberikan bobot yang berbeda berdasarkan tingkat ketertarikan pengguna. Interaksi view diberi bobot 1, addtocart diberi bobot 3, dan transaction diberi bobot 5. Semakin tinggi bobotnya, semakin besar nilai ketertarikan yang diasumsikan terhadap item tersebut.

Sebelum pemberian bobot, kolom visitorid dan itemid dikonversi menjadi tipe numerik, data yang tidak valid dihapus, dan nilai pada kolom event diseragamkan menjadi huruf kecil agar pemrosesan data lebih konsisten.

Berdasarkan hasil distribusi event pada periode P1, terdapat 322.990 interaksi view, 8.099 addtocart, dan 2.689 transaction. Contoh hasil menunjukkan bahwa event view mendapatkan bobot 1,0 sesuai dengan ketentuan yang telah ditetapkan.

Pemberian bobot ini bertujuan agar sistem rekomendasi tidak hanya memperhitungkan jumlah interaksi, tetapi juga membedakan tingkat ketertarikan pengguna. Bobot tersebut selanjutnya digunakan untuk membangun matriks interaksi sebagai dasar perhitungan KNN.

# Pembentukan Matriks Interaksi Pengguna-Item

Pada tahap ini, data interaksi pengguna pada periode P1 diolah menjadi matriks pengguna-item (user-item interaction matrix). Baris matriks merepresentasikan pengguna, sedangkan kolom merepresentasikan item. Setiap nilai pada matriks menunjukkan total bobot interaksi pengguna terhadap item tertentu.

Apabila pengguna melakukan beberapa interaksi terhadap item yang sama, bobot interaksi tersebut dijumlahkan. Matriks ini menjadi representasi preferensi pengguna yang akan digunakan untuk menghitung kemiripan antar pengguna dalam metode K-Nearest Neighbors (KNN).

In [3]:
# 2.2.3 — Membentuk matriks interaksi pengguna-item

# Menjumlahkan bobot interaksi untuk setiap pasangan pengguna dan item
interaksi_p1 = (
    df_p1.groupby(["visitorid", "itemid"])["bobot"]
    .sum()
    .reset_index()
)

# Menentukan daftar pengguna dan item secara terurut
daftar_user = np.sort(interaksi_p1["visitorid"].unique())
daftar_item = np.sort(interaksi_p1["itemid"].unique())

# Membuat pemetaan ID pengguna dan item ke indeks matriks
user_to_idx = {user: idx for idx, user in enumerate(daftar_user)}
item_to_idx = {item: idx for idx, item in enumerate(daftar_item)}

# Mengubah ID pengguna dan item menjadi indeks baris dan kolom
baris = interaksi_p1["visitorid"].map(user_to_idx).to_numpy()
kolom = interaksi_p1["itemid"].map(item_to_idx).to_numpy()
nilai = interaksi_p1["bobot"].to_numpy(dtype=float)

# Membentuk matriks sparse pengguna-item
matriks_user_item = csr_matrix(
    (nilai, (baris, kolom)),
    shape=(len(daftar_user), len(daftar_item))
)

print("Jumlah pengguna unik:", len(daftar_user))
print("Jumlah item unik:", len(daftar_item))
print("Ukuran matriks pengguna-item:", matriks_user_item.shape)
print("Jumlah pasangan pengguna-item:", matriks_user_item.nnz)
print("Kepadatan matriks: {:.6f}%".format(
    matriks_user_item.nnz /
    (matriks_user_item.shape[0] * matriks_user_item.shape[1]) * 100
))

print("\nContoh agregasi interaksi:")
display(interaksi_p1.head(10))


Jumlah pengguna unik: 185205
Jumlah item unik: 87308
Ukuran matriks pengguna-item: (185205, 87308)
Jumlah pasangan pengguna-item: 266342
Kepadatan matriks: 0.001647%

Contoh agregasi interaksi:


,visitorid,itemid,bobot
0,9,222422,1.0
1,20,176623,1.0
2,22,86411,2.0
3,36,176286,2.0
4,43,434782,1.0
5,50,331168,1.0
6,53,217218,1.0
7,59,5411,1.0
8,68,60701,1.0
9,75,96883,1.0


Pada tahap ini, bobot interaksi dijumlahkan untuk setiap pasangan pengguna dan item. Jika seorang pengguna berinteraksi dengan item yang sama beberapa kali, bobot dari seluruh interaksi tersebut digabungkan menjadi satu nilai.

Hasilnya menunjukkan terdapat 185.205 pengguna unik dan 87.308 item unik pada data training P1. Matriks interaksi yang terbentuk memiliki ukuran 185.205 × 87.308, dengan 266.342 pasangan pengguna-item yang memiliki nilai interaksi.

Kepadatan matriks hanya sebesar 0,001647%. Artinya, sebagian besar kombinasi pengguna dan item tidak memiliki interaksi. Oleh karena itu, digunakan format matriks sparse (csr_matrix) agar penyimpanan dan pemrosesan data lebih hemat memori dibandingkan matriks biasa.

Pada contoh agregasi, pasangan pengguna 22 dan item 86411 memiliki bobot 2,0. Hal ini menunjukkan bahwa total bobot interaksi pasangan tersebut setelah dijumlahkan adalah 2,0.

Matriks ini menjadi representasi perilaku pengguna terhadap item dan akan digunakan sebagai dasar untuk mencari pengguna yang memiliki pola interaksi serupa menggunakan metode KNN.

# Fungsi KNN, rekomendasi, dan metrik

In [4]:
# Fungsi KNN, rekomendasi, dan metrik
matriks_norm = normalize(matriks_user_item, norm="l2", axis=1).tocsr()   # baris dinormalisasi -> dot product = cosine
matriks_csc = matriks_user_item.tocsc()                                   # mencari user per item dengan cepat
matriks_biner = matriks_user_item.copy()
matriks_biner.data[:] = 1                                                 # untuk menghitung item bersama
matriks_biner = matriks_biner.tocsr()
n_item_user = matriks_user_item.getnnz(axis=1)


def cari_tetangga(idx_user, k=K):
    """KNN (cosine) pada user yang (1) berbagi minimal 1 item dengan user target dan
    (2) punya minimal 1 item BARU bagi user target. Tetangga tanpa item bersama (similarity 0) atau tanpa
    item baru (tidak bisa menyumbang rekomendasi) tidak ikut, sehingga k=5 benar-benar 5 tetangga yang berguna."""
    item_user = matriks_user_item[idx_user].indices
    kandidat = np.unique(np.concatenate(
        [matriks_csc.indices[matriks_csc.indptr[j]:matriks_csc.indptr[j + 1]] for j in item_user]
    ))
    kandidat = kandidat[kandidat != idx_user]
    if len(kandidat) == 0:
        return np.array([], dtype=int), np.array([])
    bersama = np.asarray(matriks_biner[kandidat][:, item_user].sum(axis=1)).ravel()
    kandidat = kandidat[n_item_user[kandidat] > bersama]
    if len(kandidat) == 0:
        return np.array([], dtype=int), np.array([])
    knn = NearestNeighbors(n_neighbors=min(k, len(kandidat)), metric="cosine", algorithm="brute")
    knn.fit(matriks_norm[kandidat])
    jarak, pos = knn.kneighbors(matriks_norm[idx_user])
    return kandidat[pos[0]], 1 - jarak[0]


def rekomendasi_knn(user, k=K, top_n=TOP_N):
    """Skor item = jumlah (similarity tetangga x bobot interaksi tetangga); item yang sudah ada di P1 dibuang."""
    idx = user_to_idx[user]
    idx_tetangga, sim = cari_tetangga(idx, k)
    if len(idx_tetangga) == 0:
        return pd.DataFrame(columns=["itemid", "skor"]), idx_tetangga, sim
    skor = np.asarray(matriks_user_item[idx_tetangga].T @ sim).ravel()
    skor[matriks_user_item[idx].indices] = 0
    kandidat = np.flatnonzero(skor > 0)
    urut = np.lexsort((daftar_item[kandidat], -skor[kandidat]))      # skor turun, tie-break itemid naik (deterministik)
    top = kandidat[urut][:top_n]
    return pd.DataFrame({"itemid": daftar_item[top], "skor": skor[top]}), idx_tetangga, sim


def hitung_metrik(daftar_rekomendasi, relevan, k=TOP_N):
    """Hit@k, NDCG@k (relevansi biner). Mengembalikan (hit, ndcg, dcg, idcg)."""
    top = list(daftar_rekomendasi)[:k]
    hit = int(any(i in relevan for i in top))
    dcg = sum(1 / np.log2(p + 2) for p, i in enumerate(top) if i in relevan)
    idcg = sum(1 / np.log2(p + 2) for p in range(min(len(relevan), k)))
    return hit, (dcg / idcg if idcg > 0 else 0.0), dcg, idcg


Pada tahap ini, dibuat fungsi KNN untuk mencari 5 pengguna dengan pola interaksi paling mirip menggunakan cosine similarity. Rekomendasi item dihitung berdasarkan skor kemiripan dan bobot interaksi tetangga, dengan mengecualikan item yang sudah pernah diakses pengguna pada P1. Kualitas rekomendasi kemudian dievaluasi menggunakan Hit@5 dan NDCG@5 berdasarkan data testing P2.

# Memilih Pengguna Target Secara Acak dan Reproduktif

Pemilihan pengguna target dilakukan secara acak dengan menggunakan random seed sebesar 42 agar proses pemilihan dapat direproduksi. Pengguna yang dipilih harus memiliki riwayat interaksi pada data training P1 dan data testing P2. Selain itu, pengguna harus mempunyai setidaknya satu item relevan pada periode P2 yang belum pernah diinteraksikan pada periode P1.

Kriteria tersebut digunakan agar sistem dapat menghasilkan rekomendasi untuk pengguna target dan mengevaluasi hasil rekomendasi berdasarkan interaksi pengguna pada periode pengujian.

In [5]:
# Memilih pengguna target secara acak dan reproduktif
item_train_per_user = df_p1.groupby("visitorid")["itemid"].apply(set).to_dict()
item_test_per_user = df_p2.groupby("visitorid")["itemid"].apply(set).to_dict()
item_train_set = set(daftar_item)


def item_relevan_user(user):
    """Item di P2 yang belum pernah diinteraksikan di P1 dan ada di kosakata item training."""
    return (item_test_per_user[user] - item_train_per_user[user]) & item_train_set


# Kandidat awal: muncul di P1 dan P2 serta punya minimal 1 item relevan
kandidat_awal = [u for u in sorted(set(item_train_per_user) & set(item_test_per_user)) if item_relevan_user(u)]
# Kandidat layak: punya >= K tetangga yang berguna, agar "k = 5" benar-benar terpenuhi
kandidat_user = [u for u in kandidat_awal if len(cari_tetangga(user_to_idx[u])[0]) >= K]
print(f"Kandidat awal: {len(kandidat_awal):,} | layak untuk KNN k={K}: {len(kandidat_user):,} "
      f"({len(kandidat_user) / len(kandidat_awal):.1%})")

rng = np.random.default_rng(SEED)                    # proses acak: seed tetap + daftar kandidat terurut
user_target = int(rng.choice(kandidat_user))
item_relevan_target = item_relevan_user(user_target)

print("Seed acak:", SEED)
print("ID pengguna target:", user_target)
print("Item di P1:", len(item_train_per_user[user_target]),
      "| item di P2:", len(item_test_per_user[user_target]),
      "| item relevan:", len(item_relevan_target))


Kandidat awal: 2,158 | layak untuk KNN k=5: 932 (43.2%)
Seed acak: 42
ID pengguna target: 130999
Item di P1: 78 | item di P2: 41 | item relevan: 34


Pengguna target dipilih secara acak menggunakan SEED = 42 agar hasilnya reproducible. Dari 2.158 kandidat awal, terdapat 932 pengguna yang memenuhi syarat untuk KNN dengan (k=5). Pengguna terpilih adalah 130999, dengan 78 item pada P1, 41 item pada P2, dan 34 item relevan yang belum pernah diinteraksikan pada P1. Item relevan tersebut digunakan untuk mengevaluasi hasil rekomendasi.

# Pencarian Tetangga Terdekat Menggunakan KNN
Pada tahap ini, metode K-Nearest Neighbors (KNN) digunakan untuk mencari lima pengguna yang memiliki pola interaksi paling mirip dengan pengguna target. Matriks interaksi pengguna-item digunakan sebagai representasi preferensi pengguna, dan kemiripan diukur dengan cosine similarity (cosine distance = 1 − similarity).

Pencarian dibatasi pada pengguna yang (1) berbagi minimal satu item dengan pengguna target dan (2) memiliki minimal satu item yang belum pernah diinteraksikan pengguna target. Pengguna tanpa item bersama selalu memiliki similarity nol sehingga tidak informatif, dan pengguna tanpa item baru tidak dapat menyumbang rekomendasi. Dengan pembatasan ini, kelima tetangga yang dikembalikan semuanya berguna untuk proses rekomendasi.

In [6]:
# KNN dan 3 item yang diiklankan untuk k = 5
rekom, idx_tetangga, sim_tetangga = rekomendasi_knn(user_target)

df_tetangga = pd.DataFrame({
    "visitorid_tetangga": daftar_user[idx_tetangga],
    "cosine_similarity": sim_tetangga,
    "item_bersama": [len(item_train_per_user[int(daftar_user[i])] & item_train_per_user[user_target])
                     for i in idx_tetangga],
})
print("Pengguna target:", user_target, "| tetangga:", len(df_tetangga), "(semua similarity > 0 dan membawa item baru)")
display(df_tetangga)


Pengguna target: 130999 | tetangga: 5 (semua similarity > 0 dan membawa item baru)


,visitorid_tetangga,cosine_similarity,item_bersama
0,904969,0.195180,1
1,945041,0.161262,5
2,19878,0.140859,3
3,888399,0.139573,3
4,1172140,0.138013,4


Berdasarkan hasil KNN dengan k = 5, ditemukan lima tetangga terdekat untuk pengguna target 130999 (78 item pada P1). Seluruh tetangga memiliki cosine similarity positif dan membawa item yang belum dimiliki pengguna target.

Tetangga terdekat adalah pengguna 904969 (similarity 0,1952), disusul 945041 (0,1613), 19878 (0,1409), 888399 (0,1396), dan 1172140 (0,1380). Pengguna 904969 hanya memiliki 2 item pada P1 dan berbagi 1 item dengan target. Similarity-nya tertinggi karena vektornya pendek, sehingga satu item bersama sudah menjadi proporsi besar dari vektornya. Sebaliknya, pengguna 945041 berbagi 5 item dengan target tetapi similarity-nya lebih rendah karena memiliki 11 item.

Nilai similarity semuanya rendah (di bawah 0,2) karena pengguna target memiliki profil yang besar sedangkan tetangga hanya berbagi sebagian kecil dari profil tersebut. Hal ini wajar pada matriks yang sangat sparse. Pada tahap berikutnya, skor rekomendasi dihitung dari similarity tetangga dikalikan bobot interaksi tetangga terhadap item.

## Pembuatan Rekomendasi

In [7]:
print("Top", TOP_N, "rekomendasi:")
display(rekom)
print("3 item yang diiklankan:", rekom["itemid"].head(3).tolist())

Top 5 rekomendasi:


,itemid,skor
0,240170,0.806312
1,455223,0.483787
2,25756,0.322525
3,208529,0.322525
4,447710,0.280432


3 item yang diiklankan: [240170, 455223, 25756]


Top-5 rekomendasi untuk pengguna 130999 adalah item 240170 (skor 0,8063), 455223 (0,4838), 25756 (0,3225), 208529 (0,3225), dan 447710 (0,2804). Tiga item yang diiklankan adalah **240170, 455223, dan 25756**.

Empat dari lima item berasal dari tetangga 945041. Item 240170 mendapat skor tertinggi karena bobot interaksi tetangga tersebut terhadap item itu besar (5, yaitu kombinasi view, add-to-cart, dan pembelian) dikalikan similarity 0,1613. Item 25756 dan 208529 memiliki skor yang sama (0,3225); urutannya ditentukan dengan aturan tie-break itemid terkecil, sehingga 25756 berada di peringkat ketiga. Item 447710 muncul dari dua tetangga (19878 dan 888399) masing-masing dengan bobot 1.

# Evaluasi Rekomendasi Menggunakan Hit@5 dan NDCG@5

Evaluasi dilakukan dengan menggunakan data testing P2 sebagai acuan relevansi. Item yang diinteraksikan pengguna pada P2 dan belum pernah diinteraksikan pada P1 diperlakukan sebagai item relevan untuk evaluasi.

Metrik Hit@5 digunakan untuk mengetahui apakah setidaknya satu item relevan muncul pada lima rekomendasi teratas. Sementara itu, NDCG@5 digunakan untuk mengukur kualitas pemeringkatan rekomendasi berdasarkan posisi item relevan dalam daftar Top 5. Kedua metrik ini membantu menilai kemampuan sistem dalam merekomendasikan item yang sesuai dengan interaksi pengguna pada periode pengujian.

In [8]:
# Hit@5 dan NDCG@5 untuk user target
hit_at_5, ndcg_at_5, dcg_at_5, idcg_at_5 = hitung_metrik(rekom["itemid"], item_relevan_target)

print("User target:", user_target)
print("Jumlah item relevan di P2:", len(item_relevan_target))
print("Top-5 rekomendasi:", rekom["itemid"].tolist())
print("Hit@5  :", hit_at_5)
print("DCG@5  :", round(dcg_at_5, 4))
print("IDCG@5 :", round(idcg_at_5, 4))
print("NDCG@5 :", round(ndcg_at_5, 4))


User target: 130999
Jumlah item relevan di P2: 34
Top-5 rekomendasi: [240170, 455223, 25756, 208529, 447710]
Hit@5  : 0
DCG@5  : 0
IDCG@5 : 2.9485
NDCG@5 : 0.0


Evaluasi dilakukan pada pengguna target 130999. Item relevan didefinisikan sebagai item yang diinteraksikan pengguna pada P2, belum pernah diinteraksikan pada P1, dan tersedia pada data training. Pengguna ini memiliki 34 item relevan, sedangkan sistem merekomendasikan lima item teratas: 240170, 455223, 25756, 208529, dan 447710.

**Hit@5 = 0.** Tidak ada satu pun dari lima item rekomendasi yang termasuk dalam 34 item relevan, sehingga rekomendasi dinyatakan tidak berhasil mengenai item yang benar-benar diinteraksikan pengguna pada P2.

**NDCG@5 = 0.** Karena tidak ada item relevan pada lima posisi teratas, nilai DCG@5 sama dengan 0. Nilai IDCG@5 sebesar 2,9485 adalah skor DCG terbaik yang mungkin dicapai, yaitu ketika kelima posisi teratas semuanya berisi item relevan (1/log₂2 + 1/log₂3 + 1/log₂4 + 1/log₂5 + 1/log₂6). Nilai itu dipakai sebagai pembagi, sehingga NDCG@5 = 0 / 2,9485 = 0.

**Mengapa hasilnya 0.**
1. Kandidat rekomendasi sangat terbatas. Skor item hanya dihitung dari item milik lima tetangga, dan jumlah item yang belum dimiliki pengguna target pada kelima tetangga tersebut paling banyak 22 item. Empat dari lima item teratas berasal dari satu tetangga (945041). Padahal katalog P1 berisi 87.308 item.
2. Item relevan sulit ditebak. Hanya 34 item dari puluhan ribu item yang diinteraksikan pada P2. Selain itu, P2 berlangsung sekitar sebulan setelah P1 sehingga minat pengguna dapat berubah.
3. Evaluasi bergantung pada interaksi yang tercatat. Item yang direkomendasikan bisa saja relevan bagi pengguna tetapi tidak sempat diinteraksikan pada P2, dan kasus ini tetap dihitung sebagai tidak relevan.
4. Evaluasi pada satu pengguna hanya menghasilkan nilai 0 atau 1 untuk Hit@5. Hasil ini belum dapat dipakai untuk menyimpulkan kualitas sistem secara keseluruhan.

Dengan demikian, nilai 0 pada pengguna target ini merupakan hasil yang wajar untuk data yang sangat sparse, dan bukan kesalahan perhitungan. Ukuran kualitas sistem sebaiknya dinilai dari rata-rata terhadap banyak pengguna.

# Evaluasi seluruh user layak dan pembanding
Satu user hanya menghasilkan nilai 0 atau 1. Hasil dari dataset-mu (932 user): User-KNN Hit@5 = 0,0311 dan NDCG@5 = 0,0171; Item-based 0,0150 dan 0,0069; Populer 0,0043 dan 0,0013. Perbandingan ini indikatif (user yang sama dipakai untuk membandingkan sekaligus melaporkan).

In [9]:
# Evaluasi seluruh user layak + pembanding. Satu user hanya menghasilkan nilai 0 atau 1.
from itertools import islice

# Pembanding 1: item terpopuler di P1 (jumlah visitor unik) yang belum dilihat user
populer = np.asarray((matriks_user_item > 0).sum(axis=0)).ravel()
urut_populer = np.lexsort((daftar_item, -populer))


def rekomendasi_populer(user, top_n=TOP_N):
    sudah = set(matriks_user_item[user_to_idx[user]].indices)
    return [int(daftar_item[j]) for j in islice((j for j in urut_populer if j not in sudah), top_n)]


# Pembanding 2: item-based (cosine item-item dari proyeksi Item); skor = sum bobot user x similarity item
matriks_item_norm = normalize(matriks_user_item.T.tocsr(), norm="l2", axis=1).tocsr()
matriks_item_norm_T = matriks_item_norm.T.tocsr()


def rekomendasi_item_based(user, top_n=TOP_N):
    idx = user_to_idx[user]
    J, w = matriks_user_item[idx].indices, matriks_user_item[idx].data
    skor = np.asarray((matriks_item_norm[J] @ matriks_item_norm_T).T @ w).ravel()
    skor[J] = 0
    kandidat = np.flatnonzero(skor > 0)
    urut = np.lexsort((daftar_item[kandidat], -skor[kandidat]))
    return daftar_item[kandidat[urut][:top_n]].tolist()


def evaluasi(users, fungsi):
    hasil = [hitung_metrik(fungsi(u), item_relevan_user(u)) for u in users]
    return pd.Series({"Hit@5": np.mean([h[0] for h in hasil]), "NDCG@5": np.mean([h[1] for h in hasil])})


tabel_eval = pd.DataFrame({
    "User-KNN (k=5)": evaluasi(kandidat_user, lambda u: rekomendasi_knn(u)[0]["itemid"].tolist()),
    "Item-based": evaluasi(kandidat_user, rekomendasi_item_based),
    "Populer": evaluasi(kandidat_user, rekomendasi_populer),
}).T
print(f"Jumlah user dievaluasi: {len(kandidat_user):,} (seluruh kandidat layak)")
display(tabel_eval.round(4))


Jumlah user dievaluasi: 932 (seluruh kandidat layak)


,Hit@5,NDCG@5
User-KNN (k=5),0.0311,0.0171
Item-based,0.0150,0.0069
Populer,0.0043,0.0013


**Evaluasi pada seluruh pengguna layak.** Pada 932 pengguna yang memiliki minimal lima tetangga informatif, rata-rata Hit@5 adalah 0,0311 (29 pengguna terkena) dan rata-rata NDCG@5 adalah 0,0171. Artinya sekitar 97% pengguna memang memperoleh Hit@5 = 0, sehingga hasil pengguna target sejalan dengan kondisi umum data ini. Sebagai pembanding, metode item-based memperoleh Hit@5 sebesar 0,0150 dan rekomendasi item terpopuler hanya 0,0043. Jadi KNN berbasis pengguna sekitar tujuh kali lebih baik daripada rekomendasi populer, walaupun nilai absolutnya rendah.